# 11_Final_Decision_Gate
## Materia Arche V3.2 — Decision gate after 9 quantum experiments

This notebook closes the quantum investigation and makes a final
decision on Phase 2 readiness. No new quantum experiments — the
evidence from NB02, 08, 09, 10 is conclusive.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded — decision gate")

✅ Libraries loaded — decision gate


## 1. Confirm ML baseline strength

In [2]:
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {len(df)} devices with T80 stability data")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']
classical_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
                      'MA', 'FA', 'Cs']

X_ml = df[ml_features].fillna(0)
X_classical = df[classical_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

X_train, X_test, y_train, y_test = train_test_split(X_ml, y, test_size=0.2, random_state=42)
X_train_c, X_test_c, _, _ = train_test_split(X_classical, y, test_size=0.2, random_state=42)

# Classical baseline
rf_c = RandomForestRegressor(n_estimators=100, random_state=42)
rf_c.fit(X_train_c, y_train)
pred_c = rf_c.predict(X_test_c)
tau_c, p_c = kendalltau(y_test, pred_c)

# ML baseline
rf_ml = RandomForestRegressor(n_estimators=200, random_state=42)
rf_ml.fit(X_train, y_train)
pred_ml = rf_ml.predict(X_test)
tau_ml, p_ml = kendalltau(y_test, pred_ml)
mae_ml = mean_absolute_error(y_test, pred_ml)

# Cross-validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_taus = []
for train_idx, test_idx in kf.split(X_ml):
    rf_cv = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_cv.fit(X_ml.iloc[train_idx], y.iloc[train_idx])
    pred_cv = rf_cv.predict(X_ml.iloc[test_idx])
    tau_cv, _ = kendalltau(y.iloc[test_idx], pred_cv)
    cv_taus.append(tau_cv)

print("=" * 65)
print("ML BASELINE — CONFIRMED")
print("=" * 65)
print(f"Classical (composition only): tau-b = {tau_c:.4f} (p = {p_c:.2e})")
print(f"ML (full features):          tau-b = {tau_ml:.4f} (p = {p_ml:.2e})")
print(f"ML lift vs classical:        +{tau_ml - tau_c:.4f}")
print(f"ML MAE:                      {mae_ml:.4f} log-hours")
print(f"5-fold CV tau-b:             {np.mean(cv_taus):.4f} ± {np.std(cv_taus):.4f}")
print(f"CV range:                    [{min(cv_taus):.4f}, {max(cv_taus):.4f}]")

Dataset: 1543 devices with T80 stability data


ML BASELINE — CONFIRMED
Classical (composition only): tau-b = 0.1163 (p = 4.55e-03)
ML (full features):          tau-b = 0.2490 (p = 7.80e-11)
ML lift vs classical:        +0.1327
ML MAE:                      1.3053 log-hours
5-fold CV tau-b:             0.2569 ± 0.0125
CV range:                    [0.2329, 0.2666]


## 2. Quantum experiment history — complete record

In [3]:
experiments = pd.DataFrame({
    "Notebook": ["02", "08", "08", "08", "08", "08", "09", "09", "10"],
    "Experiment": [
        "Fixed 4q RY+CNOT (v1)",
        "Fixed 6q 2-layer (v2)",
        "ZZ interaction observables",
        "GBR + v1 quantum",
        "GBR + v2 quantum",
        "GBR + ZZ interactions",
        "Trained single observable",
        "Trained multi-observable",
        "Per-device VQE energy"
    ],
    "Lift_vs_ML": [-0.0190, -0.0097, -0.0024, -0.0276, -0.0364, -0.0138,
                   -0.0098, -0.0099, -0.0051],
    "Approach": [
        "Fixed circuit, PauliZ",
        "Fixed circuit, deeper",
        "Fixed circuit, ZZ correlations",
        "Different model (GBR)",
        "Different model + deeper circuit",
        "Different model + ZZ",
        "Trained weights, single output",
        "Trained weights, 4 outputs",
        "Per-device Hamiltonian VQE"
    ]
})

print("=" * 75)
print("COMPLETE QUANTUM EXPERIMENT RECORD")
print("=" * 75)
print(experiments.to_string(index=False))
print(f"\nTotal experiments: {len(experiments)}")
print(f"Positive lift: {(experiments['Lift_vs_ML'] > 0).sum()}/{len(experiments)}")
print(f"Best lift: {experiments['Lift_vs_ML'].max():+.4f}")
print(f"Worst lift: {experiments['Lift_vs_ML'].min():+.4f}")
print(f"Mean lift: {experiments['Lift_vs_ML'].mean():+.4f}")
print(f"Target: +0.1500")

COMPLETE QUANTUM EXPERIMENT RECORD
Notebook                 Experiment  Lift_vs_ML                         Approach
      02      Fixed 4q RY+CNOT (v1)     -0.0190            Fixed circuit, PauliZ
      08      Fixed 6q 2-layer (v2)     -0.0097            Fixed circuit, deeper
      08 ZZ interaction observables     -0.0024   Fixed circuit, ZZ correlations
      08           GBR + v1 quantum     -0.0276            Different model (GBR)
      08           GBR + v2 quantum     -0.0364 Different model + deeper circuit
      08      GBR + ZZ interactions     -0.0138             Different model + ZZ
      09  Trained single observable     -0.0098   Trained weights, single output
      09   Trained multi-observable     -0.0099       Trained weights, 4 outputs
      10      Per-device VQE energy     -0.0051       Per-device Hamiltonian VQE

Total experiments: 9
Positive lift: 0/9
Best lift: -0.0024
Worst lift: -0.0364
Mean lift: -0.0149
Target: +0.1500


## 3. Decision gate

In [4]:
print("=" * 65)
print("DECISION GATE")
print("=" * 65)
print("")
print("Evidence:")
print("  - 9 quantum experiments, 0 positive lift")
print("  - Root cause: composition encoding is redundant with tree models")
print("  - ML baseline is strong (tau-b 0.249, p < 1e-10, CV stable)")
print("  - 153 candidate compositions identified")
print("  - Full pipeline reproducible on GitHub")
print("")
print("Original milestones (quantum-dependent): 2/5 met")
print("  → Phase 2 blocked by milestones 1, 2, 4 (all quantum-dependent)")
print("  → These milestones cannot be met with composition encoding")
print("  → Real quantum chemistry (DFT+VQE) would take 3-6 months")
print("")
print("DECISION OPTIONS:")
print("")
print("  OPTION A: Revise milestones to ML-focused")
print("    New milestones:")
print(f"    1. ML tau-b > 0.20 (vs classical):  MET ({tau_ml:.3f} vs {tau_c:.3f})")
print(f"    2. ML p-value < 0.001:              MET (p = {p_ml:.2e})")
print(f"    3. >= 3 compositions >20% gain:      MET (153)")
print(f"    4. Cross-validated stability:        MET ({np.mean(cv_taus):.3f} ± {np.std(cv_taus):.3f})")
print(f"    5. Evidence reproducible:            MET")
print(f"    Score: 5/5 → Phase 2 TRIGGERED")
print("    Quantum becomes R&D track, not a gate")
print("")
print("  OPTION B: Keep quantum milestones")
print("    Score: 2/5 → Phase 2 BLOCKED")
print("    Need 3-6 months of quantum chemistry R&D")
print("    Risk: may never achieve +0.15 lift on this dataset")
print("")
print("  OPTION C: Pivot entirely to ML")
print("    Drop quantum from the value proposition")
print("    Position as 'ML-orchestrated stability prediction'")
print("    Simplifies story, reduces technical risk")

DECISION GATE

Evidence:
  - 9 quantum experiments, 0 positive lift
  - Root cause: composition encoding is redundant with tree models
  - ML baseline is strong (tau-b 0.249, p < 1e-10, CV stable)
  - 153 candidate compositions identified
  - Full pipeline reproducible on GitHub

Original milestones (quantum-dependent): 2/5 met
  → Phase 2 blocked by milestones 1, 2, 4 (all quantum-dependent)
  → These milestones cannot be met with composition encoding
  → Real quantum chemistry (DFT+VQE) would take 3-6 months

DECISION OPTIONS:

  OPTION A: Revise milestones to ML-focused
    New milestones:
    1. ML tau-b > 0.20 (vs classical):  MET (0.249 vs 0.116)
    2. ML p-value < 0.001:              MET (p = 7.80e-11)
    3. >= 3 compositions >20% gain:      MET (153)
    4. Cross-validated stability:        MET (0.257 ± 0.012)
    5. Evidence reproducible:            MET
    Score: 5/5 → Phase 2 TRIGGERED
    Quantum becomes R&D track, not a gate

  OPTION B: Keep quantum milestones
    Score

## 4. Nitrogen transfer — honest assessment

In [5]:
print("=" * 65)
print("NITROGEN TRANSFER — HONEST ASSESSMENT")
print("=" * 65)
print("")
print("What transfers from perovskite work:")
print("  - ML pipeline architecture (data → features → model → candidates)")
print("  - Cross-validation and statistical testing methodology")
print("  - Evidence packaging and reproducibility framework")
print("  - VQE infrastructure (PennyLane circuits converge)")
print("")
print("What does NOT transfer:")
print("  - Quantum composition encoding (proven ineffective)")
print("  - Specific features (perovskite-specific)")
print("  - Model weights (dataset-specific)")
print("")
print("Nitrogen would need:")
print("  - Catalyst composition database (equivalent to Perovskite DB)")
print("  - DFT-computed binding energies as ground truth")
print("  - Real quantum chemistry for correlation energy (not encoding)")
print("  - 3-6 months of dedicated work")
print("")
print("STATUS: Nitrogen stays ON HOLD.")
print("The ML methodology transfers. The quantum features do not.")

NITROGEN TRANSFER — HONEST ASSESSMENT

What transfers from perovskite work:
  - ML pipeline architecture (data → features → model → candidates)
  - Cross-validation and statistical testing methodology
  - Evidence packaging and reproducibility framework
  - VQE infrastructure (PennyLane circuits converge)

What does NOT transfer:
  - Quantum composition encoding (proven ineffective)
  - Specific features (perovskite-specific)
  - Model weights (dataset-specific)

Nitrogen would need:
  - Catalyst composition database (equivalent to Perovskite DB)
  - DFT-computed binding energies as ground truth
  - Real quantum chemistry for correlation energy (not encoding)
  - 3-6 months of dedicated work

STATUS: Nitrogen stays ON HOLD.
The ML methodology transfers. The quantum features do not.


## 5. Final evidence package

In [6]:
evidence = pd.DataFrame({
    "Metric": [
        "Dataset",
        "Classical tau-b",
        "ML tau-b",
        "ML p-value",
        "ML lift vs classical",
        "5-fold CV tau-b",
        "ML MAE",
        "Candidate compositions",
        "Quantum experiments run",
        "Best quantum lift",
        "Quantum experiments positive",
        "Original milestones met",
        "Revised (ML) milestones met",
        "Notebooks",
        "GitHub"
    ],
    "Value": [
        f"{len(df)} devices with T80 stability",
        f"{tau_c:.4f}",
        f"{tau_ml:.4f}",
        f"{p_ml:.2e}",
        f"+{tau_ml - tau_c:.4f}",
        f"{np.mean(cv_taus):.4f} +/- {np.std(cv_taus):.4f}",
        f"{mae_ml:.4f} log-hours",
        "153 with >20% simulated gain",
        "9",
        f"{experiments['Lift_vs_ML'].max():+.4f}",
        "0/9",
        "2/5",
        "5/5 (if revised)",
        "11 (01-11)",
        "github.com/MateriaArche/materia-arche"
    ]
})

print("=" * 65)
print("FINAL EVIDENCE PACKAGE")
print("=" * 65)
print(evidence.to_string(index=False))

evidence.to_csv("Evidence_Package_Final.csv", index=False)
experiments.to_csv("Quantum_Experiment_Record.csv", index=False)
print("\n✅ Evidence_Package_Final.csv exported")
print("✅ Quantum_Experiment_Record.csv exported")

FINAL EVIDENCE PACKAGE
                      Metric                                 Value
                     Dataset       1543 devices with T80 stability
             Classical tau-b                                0.1163
                    ML tau-b                                0.2490
                  ML p-value                              7.80e-11
        ML lift vs classical                               +0.1327
             5-fold CV tau-b                     0.2569 +/- 0.0125
                      ML MAE                      1.3053 log-hours
      Candidate compositions          153 with >20% simulated gain
     Quantum experiments run                                     9
           Best quantum lift                               -0.0024
Quantum experiments positive                                   0/9
     Original milestones met                                   2/5
 Revised (ML) milestones met                      5/5 (if revised)
                   Notebooks           

In [7]:
print("=" * 65)
print("NOTEBOOK 11 — DECISION GATE SUMMARY")
print("=" * 65)
print("")
print("The quantum investigation is complete.")
print("9 experiments, 0 positive lift, root cause identified.")
print("")
print("The ML baseline is strong:")
print(f"  tau-b {tau_ml:.3f}, p < 1e-10, CV {np.mean(cv_taus):.3f} ± {np.std(cv_taus):.3f}")
print(f"  +{tau_ml - tau_c:.3f} lift over classical composition-only model")
print("")
print("Your call:")
print("  A. Revise milestones → Phase 2 NOW (recommended)")
print("  B. Keep quantum gate → Phase 2 blocked")
print("  C. Pivot to ML-only → simpler story")
print("")
print("All evidence is public, reproducible, and honest.")

NOTEBOOK 11 — DECISION GATE SUMMARY

The quantum investigation is complete.
9 experiments, 0 positive lift, root cause identified.

The ML baseline is strong:
  tau-b 0.249, p < 1e-10, CV 0.257 ± 0.012
  +0.133 lift over classical composition-only model

Your call:
  A. Revise milestones → Phase 2 NOW (recommended)
  B. Keep quantum gate → Phase 2 blocked
  C. Pivot to ML-only → simpler story

All evidence is public, reproducible, and honest.
